# Egyptian Civil Code — Full RAG pipeline (bge-m3 retrieval + fine-tuned Llama)

Connects the two parts and tests them **together on articles the model never saw in training**:

1. **Retrieve:** the user's question → bge-m3 (+ BM25) finds the most likely articles from the whole code.
2. **Generate:** your fine-tuned Llama writes a formal Arabic answer from the retrieved article(s).

Three modes are compared on the same questions:
- **oracle** — the correct article is given (upper bound; what the training notebook measures).
- **top1** — only the best retrieved article is given (this matches how the model was trained: one article per question).
- **top3** — the 3 best retrieved articles are given (the model was *not* trained on multi-article input, so this tests whether that helps or hurts).

Three **baselines** show what the fine-tuning and the article each contribute:
- **base_oracle** — the same Llama *without* the LoRA adapter, given the correct article (does fine-tuning help?).
- **lora_no_article** — the fine-tuned model with an *empty* input (is the article really needed, or did it memorise?).

The gap between *oracle* and the others shows how much **retrieval errors** cost you.

If a question cites an article number (\"ماذا تقول المادة 972؟\"), the pipeline looks that article up directly before using the embeddings (`USE_ARTICLE_LOOKUP`).

Results are reported with 95% confidence intervals **bootstrapped over articles** (questions from the same article are not independent). `answer_in_gold` is the comparable faithfulness number across modes; `answer_in_provided` depends on how much text was given, so do not compare it between top1 and top3.

**Files to upload:** `egyptian_civil_code_articles2.json` and `egyptian_lora_model.zip` (made by the training notebook; it contains the adapter and the exact held-out `test_split.json`).

> ROUGE / semantic similarity measure similarity to the reference answer, not legal correctness. Read the examples in Step 8.

## Step 1 — Install

In [ ]:
!pip install -q unsloth
!pip install -q sentence-transformers rank_bm25 rouge_score

## Step 2 — Upload the files

In [ ]:
from google.colab import files
import os
print("Upload egyptian_civil_code_articles2.json and egyptian_lora_model.zip")
uploaded = files.upload()
assert os.path.exists("/content/egyptian_civil_code_articles2.json"), "articles json missing"
assert os.path.exists("/content/egyptian_lora_model.zip"), "egyptian_lora_model.zip missing"

## Step 3 — Settings

In [ ]:
# Unsloth must be imported before transformers / peft.
from unsloth import FastLanguageModel

import json, os, random, re, zipfile
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

ARTICLES_PATH = "/content/egyptian_civil_code_articles2.json"
ZIP_PATH      = "/content/egyptian_lora_model.zip"
MAX_SEQ_LENGTH = 2048

# Retrieval (use the setting that won in the retrieval notebook)
INDEX_WITH_HEADINGS = True
RETRIEVAL_METHOD    = "hybrid (RRF)"     # "dense bge-m3", "BM25" or "hybrid (RRF)"
TOP_N, TOP_K        = 50, 3

# Evaluation
MODES  = ["oracle", "top1", "top3", "base_oracle", "lora_no_article"]   # also available: "base_no_article"
USE_ARTICLE_LOOKUP = True   # questions that cite an article number go straight to that article
N_EVAL = 300          # first pass: ~300 questions (whole articles are kept together). Use None for the final run on ALL held-out questions.
SEED   = 42

# ---- Arabic text normalisation ----
_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")
_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize_ar(text):
    t = _DIACRITICS.sub("", str(text))
    t = re.sub("[إأآٱ]", "ا", t)
    t = t.replace("ى", "ي").replace("ة", "ه").translate(_DIGITS)
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.lower().split())

# Strip common attached prefixes so "بالعقد" matches "العقد".
_PREFIXES = ("وال", "بال", "كال", "فال", "لل", "ال", "و", "ب", "ل", "ف", "ك")
def strip_prefix(w):
    for p in _PREFIXES:
        if w.startswith(p) and len(w) - len(p) >= 3:
            return w[len(p):]
    return w

def tokens(text):
    return [strip_prefix(w) for w in normalize_ar(text).split()]

# ---- Retrieval helpers ----
def index_text(article, with_headings):
    head = " / ".join(article.get("path", [])) if with_headings else ""
    return (head + "\n" + article["context"]).strip() if head else article["context"]

def rrf(*rank_arrays, k=60, top=100):
    """Reciprocal-rank fusion of several ranked lists (rows = queries)."""
    out = []
    for rows in zip(*rank_arrays):
        score = {}
        for r in rows:
            for pos, idx in enumerate(r):
                score[idx] = score.get(idx, 0.0) + 1.0 / (k + pos + 1)
        out.append(sorted(score, key=score.get, reverse=True)[:top])
    return np.array(out)

## Step 4 — Unpack the adapter and load the held-out questions

In [ ]:
extract_root = "/content/lora_adapter_extracted"
os.makedirs(extract_root, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(extract_root)

adapter_dir = None
for root, dirs, fs in os.walk(extract_root):
    if "adapter_config.json" in fs and "adapter_model.safetensors" in fs:
        adapter_dir = root
        break
assert adapter_dir, "Could not find adapter_config.json / adapter_model.safetensors in the zip"

split_path = os.path.join(adapter_dir, "test_split.json")
assert os.path.exists(split_path), (
    "test_split.json is not in the zip. Re-run Step 8 of the updated training notebook, "
    "or the test articles cannot be identified safely (they might have been seen in training).")
test_pairs = json.load(open(split_path, encoding="utf-8"))
print("Adapter:", adapter_dir)
print("Held-out test pairs:", len(test_pairs))

In [ ]:
corpus = json.load(open(ARTICLES_PATH, encoding="utf-8"))
ctx2idx = {a["context"]: i for i, a in enumerate(corpus)}

pairs = []
for p in test_pairs:
    g = ctx2idx.get(p["context"].strip())
    if g is not None:
        pairs.append({**p, "gold": g})
print(f"{len(pairs)} of {len(test_pairs)} test pairs matched an article in the corpus")

if N_EVAL and N_EVAL < len(pairs):
    by_art = {}
    for p in pairs:
        by_art.setdefault(p["gold"], []).append(p)
    art_ids = sorted(by_art)
    random.Random(SEED).shuffle(art_ids)
    chosen, n_sel = [], 0
    for a in art_ids:                      # keep all questions of an article together
        if n_sel >= N_EVAL:
            break
        chosen += by_art[a]; n_sel += len(by_art[a])
    pairs = chosen
print("Evaluating:", len(pairs), "questions from", len({p["gold"] for p in pairs}), "articles")
print(pd.Series([p["style"] for p in pairs]).value_counts().to_string())

## Step 5 — Build the retriever (bge-m3 + BM25)

In [ ]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

device = "cuda"
embedder = SentenceTransformer("BAAI/bge-m3", device=device)
embedder.max_seq_length = 1024
embedder.half()

def encode(texts):
    return embedder.encode(texts, batch_size=16, convert_to_tensor=True,
                           normalize_embeddings=True, show_progress_bar=False)

index_texts = [index_text(a, INDEX_WITH_HEADINGS) for a in corpus]
c_emb = encode(index_texts)
bm25 = BM25Okapi([tokens(t) for t in index_texts])

def retrieve(qs):
    qe = encode(qs)
    dense = torch.topk(qe.float() @ c_emb.float().T, TOP_N, dim=1).indices.cpu().numpy()
    if RETRIEVAL_METHOD == "dense bge-m3":
        return dense
    bm = np.array([np.argsort(-bm25.get_scores(tokens(q)))[:TOP_N] for q in qs])
    return bm if RETRIEVAL_METHOD == "BM25" else rrf(dense, bm, top=TOP_N)

art2idx = {a["article"]: i for i, a in enumerate(corpus)}
_ART_REF = re.compile(r"(?:الماده|ماده)\s*(?:رقم\s*)?(\d{1,4})")

def cited_articles(q):
    out = []
    for n in _ART_REF.findall(normalize_ar(q)):
        i = art2idx.get(int(n))
        if i is not None and i not in out:
            out.append(i)
    return out[:3]

def retrieve_smart(qs):
    """Normal retrieval, but articles cited by number in the question are put first."""
    base = retrieve(qs)
    if not USE_ARTICLE_LOOKUP:
        return base
    rows = []
    for q, r in zip(qs, base):
        first = cited_articles(q)
        rows.append(np.array((first + [int(x) for x in r if int(x) not in first])[:TOP_N]))
    return np.array(rows)

ret = retrieve_smart([p["question"] for p in pairs])
print("Questions that cite an article number:", sum(bool(cited_articles(p["question"])) for p in pairs))
gold_arr = np.array([p["gold"] for p in pairs])
pos = np.array([int(np.where(r == g)[0][0]) if g in r else 10**6 for r, g in zip(ret, gold_arr)])
print(f"Retrieval on these questions ({RETRIEVAL_METHOD}):  hit@1 = {(pos < 1).mean():.1%}   hit@3 = {(pos < 3).mean():.1%}   hit@{TOP_N} = {(pos < TOP_N).mean():.1%}")

## Step 6 — Load the fine-tuned Llama

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_dir, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
FastLanguageModel.for_inference(model)

# Must be identical to the prompt used for training.
alpaca_prompt = '''Below is a question about a law in Egypt (the Egyptian Civil Code, Law 131 of 1948), paired with an input that gives the text of the relevant article. Write an answer in formal Arabic that answers the question using only the article.

### Question:
{}

### Input:
{}

### Answer:
{}'''

def build_prompt(question, idxs):
    # Use as many of the given articles as fit; drop the lowest-ranked first.
    n = len(idxs)
    while True:
        ctx = "\n\n".join(corpus[i]["context"] for i in idxs[:n])
        prompt = alpaca_prompt.format(question, ctx, "")
        if n == 1 or len(tokenizer(prompt)["input_ids"]) <= MAX_SEQ_LENGTH - 256:
            return prompt, list(idxs[:n]), ctx
        n -= 1

@torch.no_grad()
def generate_answer(prompt):
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                         use_cache=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print("Model ready.")

In [ ]:
# Sanity check for the base-model baselines: switching the adapter off must change the output.
_p = alpaca_prompt.format("ما هو العقد؟", corpus[0]["context"], "")
_with = generate_answer(_p)
with model.disable_adapter():
    _without = generate_answer(_p)
print("Adapter toggle works (outputs differ):", _with != _without)
if _with == _without:
    print("WARNING: base_* baselines would be identical to the fine-tuned model. Do not use them.")


## Step 7 — Run the pipeline in each mode

In [ ]:
def faithfulness(answer, context):
    a = tokens(answer); c = set(tokens(context))
    return sum(t in c for t in a) / max(len(a), 1)

def clean_answer(text):
    # The base model has no trained stop token, so cut anything after the next prompt marker.
    return text.split("###")[0].strip()

# mode -> (which model, which article(s) go into the prompt)
MODE_CFG = {
    "oracle":          ("lora", "gold"),
    "top1":            ("lora", "top1"),
    "top3":            ("lora", "top3"),
    "base_oracle":     ("base", "gold"),
    "lora_no_article": ("lora", "none"),
    "base_no_article": ("base", "none"),
}

records = []
for mode in MODES:
    which, source = MODE_CFG[mode]
    for j, p in enumerate(tqdm(pairs, desc=mode)):
        ranks = [int(x) for x in ret[j]]
        idxs = {"gold": [p["gold"]], "top1": ranks[:1], "top3": ranks[:TOP_K], "none": []}[source]
        if idxs:
            prompt, used, ctx = build_prompt(p["question"], idxs)
        else:
            prompt, used, ctx = alpaca_prompt.format(p["question"], "", ""), [], ""
        if which == "base":
            with model.disable_adapter():
                pred = clean_answer(generate_answer(prompt))
        else:
            pred = clean_answer(generate_answer(prompt))
        gold_ctx = corpus[p["gold"]]["context"]
        records.append({
            "mode": mode, "style": p["style"], "question": p["question"],
            "gold_article": corpus[p["gold"]]["article"],
            "articles_given": [corpus[i]["article"] for i in used],
            "retrieval_hit": p["gold"] in used,      # = the correct article was in the prompt
            "reference_answer": p["answer"], "generated_answer": pred,
            "answer_in_provided": faithfulness(pred, ctx),
            "answer_in_gold": faithfulness(pred, gold_ctx),
        })
print("Generated:", len(records))


## Step 8 — Metrics and results

In [ ]:
from rouge_score import rouge_scorer
from sentence_transformers import util

class ArabicTokenizer:
    def tokenize(self, text):
        return normalize_ar(text).split()

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], tokenizer=ArabicTokenizer())
sem_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cuda")

df = pd.DataFrame(records)
sc = [scorer.score(r, p) for r, p in zip(df["reference_answer"], df["generated_answer"])]
df["rouge1"] = [s["rouge1"].fmeasure for s in sc]
df["rougeL"] = [s["rougeL"].fmeasure for s in sc]
re_ = sem_model.encode(df["reference_answer"].tolist(), convert_to_tensor=True, normalize_embeddings=True)
pe_ = sem_model.encode(df["generated_answer"].tolist(), convert_to_tensor=True, normalize_embeddings=True)
df["semantic_similarity"] = util.cos_sim(pe_, re_).diagonal().cpu().numpy()

cols = ["retrieval_hit", "rouge1", "rougeL", "semantic_similarity", "answer_in_gold", "answer_in_provided"]
order = {m: i for i, m in enumerate(MODES)}

print("=" * 78)
print("PIPELINE RESULTS (held-out articles)")
print("=" * 78)
overall = df.groupby("mode")[cols].mean().loc[MODES]
print(overall.round(4).to_string())

print("\nBy question style:")
print(df.groupby(["mode", "style"])[cols].mean().round(4).to_string())

print("\nWhen the right article WAS vs WAS NOT given to the model:")
_r = df[df["mode"].isin(["oracle", "top1", "top3"])]
t = _r.groupby(["mode", "retrieval_hit"])[["rougeL", "semantic_similarity"]].mean().round(4)
t["n"] = _r.groupby(["mode", "retrieval_hit"]).size()
print(t.to_string())
print("=" * 78)

### Confidence intervals (bootstrap over articles)
Report these in the paper instead of single numbers. Differences whose interval does not include 0 are real; the rest could be chance.

In [ ]:
def article_boot(d, col, n=2000, seed=0):
    per = d.groupby("gold_article")[col].mean().to_numpy(float)
    rng = np.random.default_rng(seed)
    bs = [rng.choice(per, len(per)).mean() for _ in range(n)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return per.mean(), lo, hi

rows = []
for m in MODES:
    d = df[df["mode"] == m]
    metrics = ["rougeL", "semantic_similarity", "answer_in_gold"] + (["retrieval_hit"] if m in ("top1", "top3") else [])
    for col in metrics:
        mean, lo, hi = article_boot(d.assign(**{col: d[col].astype(float)}), col)
        rows.append({"mode": m, "metric": col, "mean": mean, "ci_low": lo, "ci_high": hi,
                     "n_articles": d["gold_article"].nunique(), "n_questions": len(d)})
ci_df = pd.DataFrame(rows)
print(ci_df.round(3).to_string(index=False))

def paired_diff(a, b, col, n=2000, seed=0):
    A = df[df["mode"] == a].groupby("gold_article")[col].mean()
    B = df[df["mode"] == b].groupby("gold_article")[col].mean()
    diff = (A - B).dropna().to_numpy(float)
    rng = np.random.default_rng(seed)
    bs = [rng.choice(diff, len(diff)).mean() for _ in range(n)]
    return diff.mean(), *np.percentile(bs, [2.5, 97.5])

print("\nPaired differences (mode A minus mode B), rougeL and semantic similarity:")
for a, b in [("oracle", "base_oracle"), ("oracle", "top1"), ("top1", "top3"), ("oracle", "lora_no_article")]:
    if a in MODES and b in MODES:
        for col in ["rougeL", "semantic_similarity"]:
            d, lo, hi = paired_diff(a, b, col)
            print(f"  {a} - {b:16s} {col:20s} {d:+.3f}  [{lo:+.3f}, {hi:+.3f}]")


In [ ]:
# Read real examples, including cases where retrieval missed.
for mode in [m for m in ["top1"] if m in MODES]:
    sub = df[df["mode"] == mode]
    for label, rows in [("RETRIEVAL OK", sub[sub["retrieval_hit"]].head(4)),
                        ("RETRIEVAL MISSED", sub[~sub["retrieval_hit"]].head(4))]:
        for _, r in rows.iterrows():
            print("\n" + "=" * 78)
            print(f"{label} | {r['style']} | gold article {r['gold_article']} | given {r['articles_given']}")
            print("QUESTION :", r["question"])
            print("REFERENCE:", r["reference_answer"])
            print("MODEL    :", r["generated_answer"])

## Step 9 — Ask your own question

`ask("...")` retrieves the articles and writes the answer, so you can test the whole system the way a user would.

In [ ]:
def ask(question, mode="top1"):
    ranks = [int(x) for x in retrieve_smart([question])[0]]
    idxs = ranks[:1] if mode == "top1" else ranks[:TOP_K]
    prompt, used, ctx = build_prompt(question, idxs)
    answer = generate_answer(prompt)
    print("QUESTION:", question)
    print("ARTICLES GIVEN TO THE MODEL:", [corpus[i]["article"] for i in used])
    print("ANSWER:", answer)
    print("\n--- retrieved article text ---")
    print(ctx[:600])

ask("ما هي شروط صحة العقد؟")
ask("هو ينفع أطلب فسخ العقد لو الطرف التاني ما نفذش التزامه؟")

In [ ]:
df.to_csv("/content/rag_pipeline_results.csv", index=False, encoding="utf-8-sig")
ci_df.to_csv("/content/rag_pipeline_confidence_intervals.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("/content/rag_pipeline_results.csv")
files.download("/content/rag_pipeline_confidence_intervals.csv")
